<a href="https://www.kaggle.com/code/avikdas567/helios-corn-futures-climate-challenge?scriptVersionId=292042198" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/forecasting-the-future-the-helios-corn-climate-challenge/corn_climate_risk_futures_daily_master.csv
/kaggle/input/forecasting-the-future-the-helios-corn-climate-challenge/corn_regional_market_share.csv


# Imports & Settings

In [2]:
import numpy as np
import pandas as pd

from pathlib import Path
from tqdm import tqdm

import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 200)
pd.set_option("display.float_format", "{:.4f}".format)

DATA_DIR = Path("/kaggle/input/forecasting-the-future-the-helios-corn-climate-challenge")

# Load Data

In [3]:
climate = pd.read_csv(DATA_DIR / "corn_climate_risk_futures_daily_master.csv", parse_dates=["date_on"])
shares = pd.read_csv(DATA_DIR / "corn_regional_market_share.csv")

print(climate.shape)
print(shares.shape)

(320661, 41)
(95, 5)


# Merge Production Weights

In [4]:
df = climate.merge(
    shares[["region_id", "percent_country_production"]],
    on="region_id",
    how="left"
)

df["percent_country_production"] = df["percent_country_production"].fillna(0.0) / 100.0

# Define Climate Risk Columns

In [5]:
RISK_MAP = {
    "heat": [
        "climate_risk_cnt_locations_heat_stress_risk_low",
        "climate_risk_cnt_locations_heat_stress_risk_medium",
        "climate_risk_cnt_locations_heat_stress_risk_high",
    ],
    "cold": [
        "climate_risk_cnt_locations_unseasonably_cold_risk_low",
        "climate_risk_cnt_locations_unseasonably_cold_risk_medium",
        "climate_risk_cnt_locations_unseasonably_cold_risk_high",
    ],
    "wet": [
        "climate_risk_cnt_locations_excess_precip_risk_low",
        "climate_risk_cnt_locations_excess_precip_risk_medium",
        "climate_risk_cnt_locations_excess_precip_risk_high",
    ],
    "dry": [
        "climate_risk_cnt_locations_drought_risk_low",
        "climate_risk_cnt_locations_drought_risk_medium",
        "climate_risk_cnt_locations_drought_risk_high",
    ],
}

# Risk Severity Encoding

In [6]:
SEVERITY = {"low": 0.0, "medium": 0.5, "high": 1.0}

for risk, cols in RISK_MAP.items():
    df[f"climate_risk_{risk}_severity_raw"] = (
        df[cols[1]] * SEVERITY["medium"] +
        df[cols[2]] * SEVERITY["high"]
    )

# Production-Weighted Climate Risk

In [7]:
for risk in RISK_MAP.keys():
    df[f"climate_risk_{risk}_weighted"] = (
        df[f"climate_risk_{risk}_severity_raw"] *
        df["percent_country_production"]
    )

# Temporal Aggregations (CFCS Gold)

In [8]:
WINDOWS = [7, 14, 30, 60]

for risk in RISK_MAP.keys():
    for w in WINDOWS:
        df[f"climate_risk_{risk}_ma_{w}d"] = (
            df.groupby(["country_name", "region_name"])
              [f"climate_risk_{risk}_weighted"]
              .transform(lambda x: x.rolling(w, min_periods=3).mean())
        )

# Risk Momentum (Change Signals)

In [9]:
for risk in RISK_MAP.keys():
    df[f"climate_risk_{risk}_momentum_7d"] = (
        df.groupby(["country_name", "region_name"])
          [f"climate_risk_{risk}_weighted"]
          .transform(lambda x: x - x.shift(7))
    )

# Composite Stress Indices

In [10]:
df["climate_risk_temperature_stress"] = (
    df["climate_risk_heat_weighted"] +
    df["climate_risk_cold_weighted"]
)

df["climate_risk_moisture_stress"] = (
    df["climate_risk_dry_weighted"] +
    df["climate_risk_wet_weighted"]
)

df["climate_risk_total_stress"] = (
    df["climate_risk_temperature_stress"] +
    df["climate_risk_moisture_stress"]
)

# Lagged Climate Signals

In [11]:
LAGS = [7, 14, 30]

BASE_SIGNALS = [
    "climate_risk_total_stress",
    "climate_risk_temperature_stress",
    "climate_risk_moisture_stress",
]

for col in BASE_SIGNALS:
    for lag in LAGS:
        df[f"{col}_lag_{lag}d"] = (
            df.groupby(["country_name", "region_name"])[col].shift(lag)
        )

In [12]:
CLIMATE_COLS = [c for c in df.columns if c.startswith("climate_risk_")]

df[CLIMATE_COLS] = (
    df
    .groupby(["country_name", "region_name"])[CLIMATE_COLS]
    .apply(lambda x: x.ffill().bfill())
    .reset_index(drop=True)
)

# Absolute safety net
df[CLIMATE_COLS] = df[CLIMATE_COLS].fillna(0.0)

# Monthly Aggregation (Scoring Granularity)

In [13]:
monthly = (
    df
    .groupby([
        "country_name",
        "region_name",
        "date_on_year_month"
    ])
    .mean(numeric_only=True)
    .reset_index()
)

monthly["date_on"] = pd.to_datetime(monthly["date_on_year_month"], format="%Y_%m")

In [14]:
CLIMATE_FEATURES = [c for c in monthly.columns if c.startswith("climate_risk_")]

monthly[CLIMATE_FEATURES] = (
    monthly
    .groupby(["country_name", "region_name"])[CLIMATE_FEATURES]
    .apply(lambda x: x.ffill().bfill())
    .reset_index(drop=True)
)

monthly[CLIMATE_FEATURES] = monthly[CLIMATE_FEATURES].fillna(0.0)

In [15]:
missing = monthly[CLIMATE_FEATURES].isna().sum().sum()
print("Missing climate values:", missing)

Missing climate values: 0


# Local CFCS Evaluation

In [16]:
def compute_cfcs(corrs):
    sig = corrs[np.abs(corrs) >= 0.5]
    if len(sig) == 0:
        return 0
    avg_sig = min(100, np.abs(sig.mean()) * 100)
    max_sig = min(100, np.abs(sig).max() * 100)
    cnt_sig = len(sig) / len(corrs) * 100
    return 0.5 * avg_sig + 0.3 * max_sig + 0.2 * cnt_sig

# Prepare Submission File

In [17]:
META_COLS = ["date_on", "country_name", "region_name"]

CLIMATE_FEATURES = [
    c for c in monthly.columns
    if c.startswith("climate_risk_")
]

submission = monthly[META_COLS + CLIMATE_FEATURES].copy()
submission = submission.sort_values(["date_on", "country_name", "region_name"])

# Save Submission

In [18]:
submission = submission.sort_values(
    ["date_on", "country_name", "region_name"]
).reset_index(drop=True)

submission.insert(0, "ID", submission.index)

In [19]:
submission.to_csv("submission.csv", index=False)
print("submission.csv saved")
print("Shape:", submission.shape)

submission.csv saved
Shape: (10601, 56)


In [20]:
assert submission.isna().sum().sum() == 0, "Submission still has NaNs"
print("No NaNs detected. Safe to submit")

No NaNs detected. Safe to submit
